<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 5 · Solutions</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Reading a response · Solutions</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">Worked answers to the chapter's three practice challenges.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

Worked answers to Chapter 5's practice challenges. The first calls the model, and the other two use only the module from the chapter's Claude Code build.

## Setup

The same Setup as the chapter's notebook, with this notebook's own recording, and the module from the chapter's Claude Code build.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/reading-a-response--solutions.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("reading-a-response--solutions", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 2 recorded responses from claude-sonnet-5-5, recorded 2026-11-12. Set LIVE = True in Setup to call the API yourself.


In [2]:
# retry.py is the module Claude Code generated in this chapter's build, from the spec and tests in
# builds/reading-a-response/. Colab fetches it from the book's repository.
BUILD = "reading-a-response"
if "google.colab" in sys.modules:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/"
                               f"{BUILD}/generated/retry.py", "retry.py")
else:
    sys.path.insert(0, str(pathlib.Path("../../builds", BUILD, "generated").resolve()))
import retry
retry = importlib.reload(retry)
from retry import backoff_delay, call_with_retries, create_with_retries, is_retryable, retry_after

# The whole module, so you can read every function before the notebook uses it
import inspect
print(inspect.getsource(retry))

import math
import random
import time

import anthropic


def _spend_cap_reached(error):
    body = getattr(error, "body", None)
    if not isinstance(body, dict):
        return False
    inner = body.get("error")
    if not isinstance(inner, dict):
        return False
    details = inner.get("details")
    return isinstance(details, dict) and details.get("error_code") == "enforced_spend_limit_reached"


def is_retryable(error):
    if isinstance(error, anthropic.APIConnectionError):
        return True
    if isinstance(error, anthropic.APIStatusError):
        status = error.status_code
        if status == 429 and _spend_cap_reached(error):
            return False
        return status in (408, 409, 429) or status >= 500
    return False


def retry_after(error):
    response = getattr(error, "response", None)
    if response is None:
        return None
    value = response.headers.get("retry-after")
    if value is None:
        return None
    try:
        seconds = float(valu

## Challenge 1. Keep only the first item

Use a stop sequence to make the model return only the first item of a numbered list.

In [3]:
# Challenge 1. Keep only the first item of a list with a stop sequence.
import anthropic
MODEL = "claude-sonnet-5-5"
PROMPT = "List three ways to reduce support ticket volume, numbered 1. to 3., one per line, with no introduction."

# A line break is the obvious place to stop, and the API refuses it
try:
    client.messages.create(model=MODEL, max_tokens=1024, stop_sequences=["\n"],
                           messages=[{"role": "user", "content": PROMPT}])
except anthropic.BadRequestError as e:
    print("stop_sequences=['\\n'] ->", e.status_code, e.body["error"]["message"])

# The number of the second item works, since the list is numbered
reply = client.messages.create(model=MODEL, max_tokens=1024, stop_sequences=["2."],
                               messages=[{"role": "user", "content": PROMPT}])
print(reply.stop_reason, repr(reply.stop_sequence), reply.usage.output_tokens, "tokens out")
print(repr("".join(b.text for b in reply.content if b.type == "text")))

stop_sequences=['\n'] -> 400 stop_sequences: each stop sequence must contain non-whitespace
stop_sequence '2.' 60 tokens out
'1. Build a searchable self-service knowledge base with up-to-date FAQs, how-to guides, and troubleshooting articles.\n'


A line break is the natural place to stop, and the API refused it with a 400, "stop_sequences: each stop sequence must contain non-whitespace", as of November 2026. The number of the second item works, since the list is numbered. The reply stopped at `2.` after 60 output tokens, and its text is the first item alone, ending with the line break before the stop sequence.

## Challenge 2. Which statuses are retried

Run `is_retryable` on a status error for each code in Anthropic's error list and see which it retries.

In [4]:
# Challenge 2. Which statuses does is_retryable retry?
import anthropic, httpx2
request = httpx2.Request("POST", "https://api.anthropic.com/v1/messages")
for status in [400, 401, 402, 403, 404, 408, 409, 413, 429, 500, 504, 529]:
    response = httpx2.Response(status, request=request, json={"type": "error", "error": {"type": "x", "message": "x"}})
    error = anthropic.APIStatusError("Written for the challenge.", response=response, body=response.json())
    print(status, is_retryable(error))

400 False
401 False
402 False
403 False
404 False
408 True
409 True
413 False
429 True
500 True
504 True
529 True


The request errors, 400, 401, 402, 403, 404 and 413, come back `False`. Timeouts (408), conflicts (409), rate limits (429) and the server errors 500, 504 and 529 come back `True`. The split follows the published error types, where the first group describes the request or the account and the second a condition that can clear on its own.

## Challenge 3. Give up on a long wait

`call_with_retries` waits as long as `retry-after` says. Show what it does with a 429 that asks for 600 seconds, then add a limit that raises the error instead of waiting longer than `max_wait`, with a test.

In [5]:
# Challenge 3. Give up instead of sleeping longer than max_wait.
import httpx2

def long_wait_error(seconds):
    request = httpx2.Request("POST", "https://api.anthropic.com/v1/messages")
    response = httpx2.Response(429, request=request, headers={"retry-after": str(seconds)},
                               json={"type": "error", "error": {"type": "rate_limit_error", "message": "Simulated."}})
    return anthropic.RateLimitError("Simulated.", response=response, body=response.json())

# As generated, call_with_retries sleeps for whatever retry-after says
pending, slept = [long_wait_error(600)], []

def fails_once():
    if pending:
        raise pending.pop(0)
    return "done"

print(call_with_retries(fails_once, sleep=slept.append), "after sleeping", slept, "seconds")


def call_with_max_wait(fn, max_wait=60.0, **options):
    """call_with_retries, but raise the error at once when the wait would exceed max_wait."""
    def guard(number, error, delay):
        if delay > max_wait:
            raise error
    return call_with_retries(fn, on_retry=guard, **options)


def test_a_wait_longer_than_max_wait_raises_at_once():
    pending, slept = [long_wait_error(600)], []
    def fn():
        if pending:
            raise pending.pop(0)
        return "done"
    try:
        call_with_max_wait(fn, max_wait=60, sleep=slept.append)
    except anthropic.RateLimitError:
        assert slept == []
        return
    raise AssertionError("the call slept and retried")

test_a_wait_longer_than_max_wait_raises_at_once()
print("pass: a 600-second retry-after is raised with no wait when max_wait is 60")

done after sleeping [600.0] seconds
pass: a 600-second retry-after is raised with no wait when max_wait is 60


As generated, the module slept 600 seconds, recorded here by the stand-in sleep, before its second attempt. `call_with_max_wait` adds the limit without changing `retry.py`, by passing an `on_retry` that raises the error when the delay is over `max_wait`. The test passes, with the 429 raised and no wait recorded. A request that serves a person can rarely wait ten minutes, so the caller decides how long is too long.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>